# Caminho Livre CV — treino do YOLO11n (Google Colab)

GS FIAP 2026.2 · Applied Computer Vision · fine-tuning do YOLO11n no
[ROD-Dataset](https://huggingface.co/datasets/Abtinzandi/Obstacle-Detection-Dataset-YOLO)
(24.326 imagens, 25 classes).

**Runtime:** Ambiente de execução → Alterar tipo → **GPU T4**. Checkpoints vão para o Google Drive,
então o treino sobrevive a uma desconexão (use a célula de retomada).

In [ ]:
!nvidia-smi
%pip install -q ultralytics huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/caminho-livre-cv/runs'
DATA_ROOT = '/content/rod'
import os; os.makedirs(RUN_DIR, exist_ok=True)

## 1. Preparar o dataset
Cópia fiel de `scripts/prepare_dataset.py` do repositório.

In [ ]:
%%writefile prepare_dataset.py
"""Baixa o ROD-Dataset (Hugging Face), gera o data.yaml do projeto e as estatísticas por classe.

Uso:
    python scripts/prepare_dataset.py --root data/rod --download
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import yaml

REPO_ID = "Abtinzandi/Obstacle-Detection-Dataset-YOLO"
SPLITS = ("train", "valid", "test")

# Ordem e nomes idênticos ao data.yaml original, exceto o typo "Teraffic Barrel" (id 20) corrigido.
CLASS_NAMES: list[str] = [
    "Bike", "Building", "Car", "Person", "Stairs", "Traffic sign", "Electrical Pole", "Road",
    "Motorcycle", "Dustbin", "Dog", "Manhole", "Tree", "Guard rail", "Pedestrian crosswalk",
    "Truck", "Bus", "Bench", "Traffic Cone", "Fire hydrant", "Traffic Barrel", "Plant Pot",
    "Electrical Box", "Chair", "Bicycle Rack",
]


def write_data_yaml(root: Path, out: Path) -> Path:
    data = {
        "path": str(Path(root).resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return out


def class_stats(root: Path) -> dict[str, dict[str, int]]:
    stats = {name: {split: 0 for split in SPLITS} for name in CLASS_NAMES}
    for split in SPLITS:
        for label in (Path(root) / split / "labels").glob("*.txt"):
            for line in label.read_text(encoding="utf-8").splitlines():
                parts = line.split()
                if parts:
                    stats[CLASS_NAMES[int(parts[0])]][split] += 1
    return stats


def write_stats_csv(stats: dict[str, dict[str, int]], out: Path) -> None:
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["class_id", "class_name", *SPLITS, "total"])
        for i, name in enumerate(CLASS_NAMES):
            row = [stats[name][s] for s in SPLITS]
            w.writerow([i, name, *row, sum(row)])


def download(root: Path) -> None:
    from huggingface_hub import snapshot_download

    snapshot_download(repo_id=REPO_ID, repo_type="dataset", local_dir=str(root))


def main(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--root", type=Path, default=Path("data/rod"))
    p.add_argument("--download", action="store_true")
    p.add_argument("--stats-out", type=Path, default=Path("docs/assets/class_stats.csv"))
    args = p.parse_args(argv)
    if args.download:
        download(args.root)
    yaml_path = write_data_yaml(args.root, args.root / "data_cl.yaml")
    stats = class_stats(args.root)
    write_stats_csv(stats, args.stats_out)
    total = sum(sum(v.values()) for v in stats.values())
    print(f"data.yaml: {yaml_path}\ncaixas: {total}\nestatísticas: {args.stats_out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
!python prepare_dataset.py --root $DATA_ROOT --download --stats-out /content/class_stats.csv
!ls $DATA_ROOT/train/images | wc -l; ls $DATA_ROOT/valid/images | wc -l; ls $DATA_ROOT/test/images | wc -l

## 2. Treino
YOLO11n a partir dos pesos COCO · `imgsz=640` · 50 épocas · early stopping `patience=15` · `seed=0`.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=f'{DATA_ROOT}/data_cl.yaml', epochs=50, imgsz=640, patience=15, seed=0,
            project=RUN_DIR, name='y11n', exist_ok=True)

### 2b. Retomar (só se o Colab desconectou)
Rode as células 1–3 e a preparação de novo, depois esta.

In [ ]:
from ultralytics import YOLO
YOLO(f'{RUN_DIR}/y11n/weights/last.pt').train(resume=True)

## 3. Avaliação no split **test** (não visto no treino)

In [ ]:
import json
from ultralytics import YOLO
best = YOLO(f'{RUN_DIR}/y11n/weights/best.pt')
m = best.val(data=f'{DATA_ROOT}/data_cl.yaml', split='test', imgsz=640, project=RUN_DIR, name='y11n_test', exist_ok=True)
names = best.names
per_class = {names[int(c)]: round(float(ap), 4) for c, ap in zip(m.box.ap_class_index, m.box.ap50)}
metrics = {
    'split': 'test', 'model': 'yolo11n', 'imgsz': 640,
    'map50': round(float(m.box.map50), 4), 'map50_95': round(float(m.box.map), 4),
    'precision': round(float(m.box.mp), 4), 'recall': round(float(m.box.mr), 4),
    'ap50_per_class': per_class,
    'speed_ms': m.speed,
}
json.dump(metrics, open(f'{RUN_DIR}/test_metrics.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## 4. Empacotar artefatos para o repositório

In [ ]:
import shutil, os
pack = '/content/artefatos'; os.makedirs(pack, exist_ok=True)
tr, te = f'{RUN_DIR}/y11n', f'{RUN_DIR}/y11n_test'
for src in [f'{tr}/weights/best.pt', f'{tr}/results.csv', f'{tr}/results.png', f'{tr}/args.yaml',
            f'{te}/confusion_matrix_normalized.png', f'{te}/BoxPR_curve.png', f'{te}/val_batch0_pred.jpg',
            f'{RUN_DIR}/test_metrics.json', '/content/class_stats.csv']:
    if os.path.exists(src):
        shutil.copy(src, pack)
    else:
        print('ausente:', src)
shutil.make_archive('/content/drive/MyDrive/caminho-livre-cv/artefatos', 'zip', pack)
print(os.listdir(pack))